In [2]:
import sys
sys.path.append('../')

import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

from data.synthetic.generate_synthetic_data import generate_synthetic_batches

np.random.seed(42)

In [5]:
df = generate_synthetic_batches(n_normal=500, n_fraud=50)
df.head()

,id,staff_id,product_name,flour_used_kg,expected_units,actual_units,variance_percent,is_fraud,created_at
0,f757e261-7444-4e7c-99ab-5d9008833c3d,staff_6,Naan,3.71,148,148,0.00,False,2026-04-03 12:10:08.710106
1,d9ba8f57-a5f2-45fc-9b06-d2360638983e,staff_6,Rusk,7.25,290,288,-0.69,False,2026-02-27 12:10:08.710106
2,007a8a0b-5f36-45f2-9165-b325be7aaad2,staff_8,Naan,6.72,269,268,-0.37,False,2026-07-09 12:10:08.710106
3,87f55464-33fd-4eb0-8055-c92d7c6237f0,staff_8,Cake Slice,4.88,195,195,0.00,False,2026-08-16 12:10:08.710106
4,130ac523-e706-4ae8-9b2c-cd21dda4c617,staff_6,Cake Slice,4.59,184,173,-5.98,True,2026-06-27 12:10:08.710106


In [6]:
TOLERANCE = 2.0
df['rule_based_flag'] = df['variance_percent'].abs() > TOLERANCE

rb_precision = precision_score(df['is_fraud'], df['rule_based_flag'])
rb_recall = recall_score(df['is_fraud'], df['rule_based_flag'])
rb_f1 = f1_score(df['is_fraud'], df['rule_based_flag'])

print(f'Rule-Based -> Precision: {rb_precision:.3f}  Recall: {rb_recall:.3f}  F1: {rb_f1:.3f}')

Rule-Based -> Precision: 0.658  Recall: 1.000  F1: 0.794


In [7]:
features = ['variance_percent', 'flour_used_kg', 'actual_units', 'expected_units']
X = df[features].values

iso = IsolationForest(contamination=0.09, random_state=42, n_estimators=150)
iso.fit(X)

df['iso_prediction'] = iso.predict(X)
df['iso_flag'] = df['iso_prediction'] == -1

iso_precision = precision_score(df['is_fraud'], df['iso_flag'])
iso_recall = recall_score(df['is_fraud'], df['iso_flag'])
iso_f1 = f1_score(df['is_fraud'], df['iso_flag'])

print(f'Isolation Forest -> Precision: {iso_precision:.3f}  Recall: {iso_recall:.3f}  F1: {iso_f1:.3f}')

Isolation Forest -> Precision: 0.720  Recall: 0.720  F1: 0.720


In [8]:
comparison = pd.DataFrame({
    'Method': ['Rule-Based (±2%)', 'Isolation Forest'],
    'Precision': [rb_precision, iso_precision],
    'Recall': [rb_recall, iso_recall],
    'F1 Score': [rb_f1, iso_f1],
})
comparison

,Method,Precision,Recall,F1 Score
0,Rule-Based (±2%),0.657895,1.00,0.793651
1,Isolation Forest,0.720000,0.72,0.720000
